In [1]:
import pandas as pd

train = pd.read_parquet("../data/UNSW_NB15_training-set.parquet")
test = pd.read_parquet("../data/UNSW_NB15_testing-set.parquet")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nColumns:", list(train.columns))
print("\nMissing values:", train.isna().sum().sum())

for col in ["label", "attack_cat"]:
    if col in train.columns:
        print(f"\n{col} counts:")
        print(train[col].value_counts())
    else:
        print(f"\n(no column named {col})")

train.head(3)

Train shape: (175341, 36)
Test shape: (82332, 36)

Columns: ['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'attack_cat', 'label']

Missing values: 0

label counts:
label
1    119341
0     56000
Name: count, dtype: int64

attack_cat counts:
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.121478,tcp,-,FIN,6,4,258,172,74.087486,14158.942383,...,0,0,1,1,0,0,0,0,Normal,0
1,0.649902,tcp,-,FIN,14,38,734,42014,78.473373,8395.112305,...,0,0,1,1,0,0,0,0,Normal,0
2,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,1572.271851,...,0,0,1,1,0,0,0,0,Normal,0


In [2]:
print("TRAIN attack types:")
print(train["attack_cat"].value_counts())

print("\nTEST attack types:")
print(test["attack_cat"].value_counts())

print("\nTRAIN normal vs attack (label 0/1):")
print(train["label"].value_counts())

print("\nText (categorical) columns:")
print(train.select_dtypes(exclude="number").columns.tolist())

for c in ["proto", "service", "state"]:
    print(f"{c}: {train[c].nunique()} unique values")

TRAIN attack types:
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64

TEST attack types:
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64

TRAIN normal vs attack (label 0/1):
label
1    119341
0     56000
Name: count, dtype: int64

Text (categorical) columns:
['proto', 'service', 'state', 'attack_cat']
proto: 133 unique values
service: 13 unique values
state: 9 unique values


In [5]:
import joblib
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

cat_cols = ["proto", "service", "state"]

X_full = train.drop(columns=["label", "attack_cat"])
y_full = train["label"]
cat_full = train["attack_cat"]

X_test = test.drop(columns=["label", "attack_cat"])
y_test = test["label"]
cat_test = test["attack_cat"]

# 80/20 train/validation split, keeping every attack type balanced
X_train, X_val, y_train, y_val, cat_train, cat_val = train_test_split(
    X_full, y_full, cat_full,
    test_size=0.2, stratify=cat_full, random_state=42
)

# Turn text columns into numbers (fit on train only)
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
enc.fit(X_train[cat_cols])
for part in (X_train, X_val, X_test):
    part[cat_cols] = enc.transform(part[cat_cols])

# Group attack types for the per-attack report
group_map = {
    "Normal": "Normal",
    "DoS": "DoS/DDoS",
    "Reconnaissance": "Port scan/Recon",
    "Backdoor": "Malware-like",
    "Shellcode": "Malware-like",
    "Worms": "Malware-like",
    "Generic": "Generic",
    "Exploits": "Exploits",
    "Fuzzers": "Fuzzers",
    "Analysis": "Analysis",
}
group_test = cat_test.map(group_map)

joblib.dump(enc, "../models/encoder.joblib")

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("\nAttack share in train:", round(y_train.mean() * 100, 1), "%")
print("\nTest groups:")
print(group_test.value_counts())

Train: (140272, 34) | Val: (35069, 34) | Test: (82332, 34)

Attack share in train: 68.1 %

Test groups:
attack_cat
Normal             37000
Generic            18871
Exploits           11132
Fuzzers             6062
DoS/DDoS            4089
Port scan/Recon     3496
Malware-like        1005
Analysis             677
Name: count, dtype: int64


In [6]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

t0 = time.time()
rf.fit(X_train, y_train)
print(f"Training time: {time.time() - t0:.1f}s\n")

def report(name, X, y):
    t = time.time()
    pred = rf.predict(X)
    ms = (time.time() - t) / len(X) * 1000
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    print(f"--- {name} ---")
    print(f"Accuracy : {accuracy_score(y, pred):.4f}")
    print(f"Precision: {precision_score(y, pred):.4f}")
    print(f"Recall   : {recall_score(y, pred):.4f}")
    print(f"FPR      : {fp / (fp + tn):.4f}")
    print(f"Speed    : {ms:.4f} ms per record\n")
    return pred

report("VALIDATION", X_val, y_val)
pred_test = report("TEST", X_test, y_test)

# Per attack group (attack groups = detection rate, Normal = correctly left alone)
df = pd.DataFrame({"group": group_test.values,
                   "correct": (pred_test == y_test.values)})
print("Per group on TEST:")
print(df.groupby("group")["correct"].agg(rate="mean", rows="count").round(3))

Training time: 3.5s

--- VALIDATION ---
Accuracy : 0.9519
Precision: 0.9550
Recall   : 0.9753
FPR      : 0.0979
Speed    : 0.0034 ms per record

--- TEST ---
Accuracy : 0.8663
Precision: 0.8179
Recall   : 0.9739
FPR      : 0.2656
Speed    : 0.0025 ms per record

Per group on TEST:
                  rate   rows
group                        
Analysis         0.997    677
DoS/DDoS         0.994   4089
Exploits         0.988  11132
Fuzzers          0.840   6062
Generic          0.999  18871
Malware-like     0.976   1005
Normal           0.734  37000
Port scan/Recon  0.997   3496


In [7]:
import numpy as np
from sklearn.metrics import recall_score, precision_score, accuracy_score, confusion_matrix

proba_val = rf.predict_proba(X_val)[:, 1]
proba_test = rf.predict_proba(X_test)[:, 1]

print("thr   | VAL fpr  recall | TEST fpr  recall  precision  accuracy")
for thr in [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    pv = (proba_val >= thr).astype(int)
    pt = (proba_test >= thr).astype(int)
    tnv, fpv, fnv, tpv = confusion_matrix(y_val, pv).ravel()
    tnt, fpt, fnt, tpt = confusion_matrix(y_test, pt).ravel()
    print(f"{thr:<5} | {fpv/(fpv+tnv):.3f}   {recall_score(y_val, pv):.3f}  | "
          f"{fpt/(fpt+tnt):.3f}    {recall_score(y_test, pt):.3f}   "
          f"{precision_score(y_test, pt):.3f}      {accuracy_score(y_test, pt):.3f}")

thr   | VAL fpr  recall | TEST fpr  recall  precision  accuracy
0.3   | 0.153   0.994  | 0.381    0.994   0.762      0.825
0.5   | 0.100   0.977  | 0.272    0.975   0.814      0.864
0.6   | 0.072   0.961  | 0.195    0.957   0.858      0.889
0.7   | 0.043   0.936  | 0.117    0.932   0.907      0.910
0.8   | 0.021   0.909  | 0.060    0.905   0.948      0.920
0.9   | 0.010   0.870  | 0.025    0.866   0.977      0.915
0.95  | 0.007   0.840  | 0.015    0.829   0.985      0.899
